# 04 — Results and Feedback

Parse available solver results, inspect the manual FEA gate, and build comparison and feedback artifacts.

In [ ]:
from pathlib import Path
import json
import logging
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import src.interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

sample_id = "00689964"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
STATE_A_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "01_dataset_original"
STATE_B_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"
STATE_C_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "05_post_fea_revision"
MANUAL_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "04_manual_freecad_fea"
COMPARISON_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "03_comparison"
STATE_A_VIEWS_DIR = STATE_A_DIR / "views"
STATE_B_VIEWS_DIR = STATE_B_DIR / "views"
STATE_C_VIEWS_DIR = STATE_C_DIR / "views"

print("[SETUP] complete")
print(f"  → sample_id : {sample_id}")
print(f"  → state A   : {STATE_A_DIR}")
print(f"  → state B   : {STATE_B_DIR}")
print(f"  → manual    : {MANUAL_DIR}")
print(f"  → compare   : {COMPARISON_DIR}")

In [ ]:
print("[STAGE] solver results and comparison inputs")
original_prompt_path = STATE_A_DIR / "original_prompt.txt"
original_code_path = STATE_A_DIR / "database_original_code.py"
revision_prompt_path = STATE_B_DIR / "fea_revision_prompt.txt"
revision_code_path = STATE_B_DIR / "fea_revision_code.py"
revision_change_log_path = STATE_B_DIR / "fea_revision_change_log.json"
load_case_path = STATE_B_DIR / "load_case.json"
manual_report_path = MANUAL_DIR / "fea_report.json"
screenshots_dir = MANUAL_DIR / "screenshots"
parsed_results_path = STATE_B_DIR / "parsed_results.json"
geometry_metrics_path = COMPARISON_DIR / "geometry_metrics.json"
prompt_diffs_path = COMPARISON_DIR / "prompt_and_code_diffs.md"
change_log_summary_path = COMPARISON_DIR / "change_log_summary.md"
state_abc_grid_path = COMPARISON_DIR / "state_abc_grid.png"
final_report_path = COMPARISON_DIR / "final_experiment_report.md"

if parsed_results_path.exists():
    parsed_results_payload = json.loads(parsed_results_path.read_text(encoding="utf-8"))
else:
    parsed_results_payload = None
print(f"  ✓ parsed results : {parsed_results_path if parsed_results_path.exists() else '<missing>'}")
print(f"  ✓ prompt paths   : {original_prompt_path} | {revision_prompt_path}")
print(f"  ✓ code paths     : {original_code_path} | {revision_code_path}")
print(f"  ✓ change log     : {revision_change_log_path}")
assert original_prompt_path.exists()
assert original_code_path.exists()
assert revision_prompt_path.exists()
assert revision_code_path.exists()

In [ ]:
print("[STAGE] geometry metrics and A/B comparison artifacts")
geometry_metrics = api.compute_geometry_metrics(
    {
        "state_a": STATE_A_DIR / "original.stl",
        "state_b": STATE_B_DIR / "fea_revision.stl",
    },
    geometry_metrics_path,
    force=False,
) if not geometry_metrics_path.exists() else api.load_geometry_metrics(geometry_metrics_path)
geometry_metrics_md_path = api.build_geometry_metrics_markdown(geometry_metrics, COMPARISON_DIR / "geometry_metrics.md", force=False) if not (COMPARISON_DIR / "geometry_metrics.md").exists() else (COMPARISON_DIR / "geometry_metrics.md")
original_prompt = original_prompt_path.read_text(encoding="utf-8")
revision_prompt = revision_prompt_path.read_text(encoding="utf-8")
original_code = original_code_path.read_text(encoding="utf-8")
revision_code = revision_code_path.read_text(encoding="utf-8")
prompt_diffs_written = api.build_prompt_and_code_diffs_report(
    original_prompt,
    revision_prompt,
    original_code,
    revision_code,
    prompt_diffs_path,
    force=False,
) if not prompt_diffs_path.exists() else prompt_diffs_path
change_log_summary_written = api.build_change_log_summary(
    json.loads(revision_change_log_path.read_text(encoding="utf-8")),
    change_log_summary_path,
    force=False,
) if not change_log_summary_path.exists() else change_log_summary_path
print(f"  ✓ geometry metrics : {geometry_metrics_path}")
print(f"  ✓ geometry markdown : {geometry_metrics_md_path}")
print(f"  ✓ prompt diffs      : {prompt_diffs_written}")
print(f"  ✓ change log summary: {change_log_summary_written}")
assert "states" in geometry_metrics
assert len(geometry_metrics["states"]) >= 2
assert prompt_diffs_path.exists()
assert change_log_summary_path.exists()

In [ ]:
print("[STAGE] manual gate and State C promotion")
load_case = api.LoadCase(**json.loads(load_case_path.read_text(encoding="utf-8")))
manual_report = json.loads(manual_report_path.read_text(encoding="utf-8")) if manual_report_path.exists() else {}
evidence_paths = sorted(screenshots_dir.glob("*.png")) if screenshots_dir.exists() else []
validation = api.validate_manual_fea_completion(manual_report, evidence_paths)
print(f"  ✓ manual complete : {validation['is_complete']}")
print(f"  ✓ missing fields  : {validation['missing_fields']}")
print(f"  ✓ missing paths   : {validation['missing_evidence_paths']}")
if validation["is_complete"] and not (STATE_C_DIR / "post_fea.step").exists():
    pipeline_config = api.PipelineConfig(
        config_name="config_gpt_5_4_mini.yaml",
        config_path=MODULE_ROOT / "src" / "copied_from_cadcodeverify" / "configs" / "config_gpt_5_4_mini.yaml",
        output_root=OUTPUT_ROOT,
        force=False,
    )
    post_revision = api.revise_code_after_fea(
        revision_code,
        load_case,
        manual_report,
        evidence_paths,
        pipeline_config,
    )
    post_revision_code = post_revision.code_path.read_text(encoding="utf-8")
    post_exec = api.execute_and_export_post_fea_revision_cadquery(post_revision_code, STATE_C_DIR, force=False)
    state_abc_grid = api.build_state_abc_grid(
        STATE_A_VIEWS_DIR,
        STATE_B_VIEWS_DIR,
        STATE_C_VIEWS_DIR,
        state_abc_grid_path,
        force=False,
    )
    final_report = api.build_final_experiment_report(
        sample_id,
        COMPARISON_DIR,
        geometry_metrics,
        prompt_diffs_written,
        change_log_summary_written,
        state_abc_grid,
        report_summary={
            "manual_gate_complete": validation["is_complete"],
            "solver_status": parsed_results_payload.get("overall_pass") if parsed_results_payload else None,
        },
        force=False,
    )
    print(f"  ✓ post-fea prompt : {post_revision.prompt_path}")
    print(f"  ✓ post-fea code   : {post_revision.code_path}")
    print(f"  ✓ state C step    : {post_exec['step_path']}")
    print(f"  ✓ state C stl     : {post_exec['stl_path']}")
    print(f"  ✓ state ABC grid  : {state_abc_grid}")
    print(f"  ✓ final report    : {final_report}")
else:
    print("  ✗ manual evidence incomplete; State C remains blocked.")
assert isinstance(validation, dict)
assert "is_complete" in validation
assert "missing_fields" in validation

In [ ]:
print("[STAGE] failure case")
try:
    api.build_state_abc_grid(
        MODULE_ROOT / "notebooks" / "missing-a",
        MODULE_ROOT / "notebooks" / "missing-b",
        MODULE_ROOT / "notebooks" / "missing-c",
        COMPARISON_DIR / "should_not_exist.png",
        force=False,
    )
except Exception as exc:
    print(f"  ✓ error type : {type(exc).__name__}")
    print(f"  ✓ message    : {exc}")

## Summary

- The notebook parses available results and shows the manual FEA gate status.
- Comparison artifacts are built through the public reporting functions.
- State C promotion is only attempted when the manual gate is complete.